In [1]:
import os
import pickle
import numpy as np
from tqdm.notebook import tqdm

from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Dropout, add

In [ ]:
from google.colab import files
files.upload()


In [3]:
import os
import zipfile

# Tạo thư mục Kaggle và di chuyển file kaggle.json vào đó
os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/

# Cấp quyền truy cập cho kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

# Kiểm tra xem Kaggle API đã hoạt động chưa
!kaggle datasets list


ref                                                          title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
algozee/teenager-menthal-healy                               Social Media Impact on Teen Mental Health                16190  2026-04-05 08:04:21.823000           8608        201                1  
nalisha/job-salary-prediction-dataset                        Job Salary Prediction Dataset                          3144815  2026-03-16 19:54:33.843000          14125        317                1  
arfeenkabir/customer-purchase-behavior-analysis              Customer Purchase Behavior Analysis                      77715  2026-04-16 20:10:57.803000            777         22                1  
sharmajicoder/a

In [4]:
#!/bin/bash
!kaggle datasets download adityajn105/flickr8k


Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
100% 1.04G/1.04G [00:05<00:00, 221MB/s]



In [5]:
# Giải nén file zip
dataset_path = "/content/flickr8k.zip"  # Thay tên file nếu cần
with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
    zip_ref.extractall("/content/sample_data/img_cap")  # Thư mục chứa dữ liệu sau khi giải nén

# Kiểm tra dữ liệu đã được giải nén chưa



In [6]:
import os
os.listdir("/content/sample_data/img_cap")

['captions.txt', 'Images']

In [7]:
BASE_DIR = "/content/sample_data/img_cap/"

### Extract Image Feature

In [8]:
model = VGG16()
model = Model(inputs=model.inputs, outputs=model.layers[-2].output)
print(model.summary())

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,260,544 (512.16 MB)

 Trainable params: 134,260,544 (512.16 MB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
features ={}
directory = os.path.join(BASE_DIR, "Images")

for img_name in tqdm(os.listdir(directory)):
    img_path =  directory + '/' + img_name
    image = load_img(img_path, target_size=(224,224))
    #convert img to np.array
    image = img_to_array(image)
    #reshape
    image= image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
    #preprocess
    image = preprocess_input(image)
    # extract feature
    feature = model.predict(image, verbose=0)
    #get ID image
    image_id = img_name.split('.')[0]
    #store feature
    features[image_id] = feature

  0%|          | 0/8091 [00:00<?, ?it/s]

In [ ]:
len(features)

In [ ]:
WORKING_DIR = "/content/sample_data/working"
pickle.dump(features, open(os.path.join(WORKING_DIR, 'features.pkl'),'wb'))

In [ ]:
with open(os.path.join(WORKING_DIR, 'features.pkl'), 'rb') as f:
    features = pickle.load(f)

Load caption data

In [ ]:
with open(os.path.join(BASE_DIR, 'captions.txt'),'r') as f:
    next(f)
    captions_doc = f.read()


In [ ]:
captions_doc

In [ ]:
#create mapping of image to caption
mapping = {}
for line in tqdm(captions_doc.split('\n')):
    #split the line by command
    tokens = line.split(',')
    if len(line) < 2:
        continue
    image_id, caption = tokens[0], tokens[1:]
    #remove extension from img ID
    image_id = image_id.split('.')[0]
    #convert caption list to string
    caption = " ".join(caption)
    #create list if needed
    if image_id not in mapping:
        mapping[image_id]= []
    mapping[image_id].append(caption)

In [ ]:
len(mapping)

Preprocess text data

In [ ]:
def clean(mapping):
    for key, captions in mapping.items():
        for i in range(len(captions)):
            #take 1 caption
            caption = captions[i]
            caption = caption.lower()
            caption = caption.replace('[^A-Za-z]', '')
            caption = caption.replace('\s+', ' ')
            caption = 'startseq ' + " ".join([word for word in caption.split() if len(word) > 1]) + ' endseq'
            captions[i] = caption

In [ ]:
#before preprocess  of text
mapping['1000268201_693b08cb0e']

In [ ]:
# preprocess the text
clean(mapping)

In [ ]:
mapping['1000268201_693b08cb0e']

In [ ]:
all_captions = []
for key in mapping:
    for caption in mapping[key]:
        all_captions.append(caption)


In [ ]:
len(all_captions)

In [ ]:
all_captions[:10]

In [ ]:
#tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocal_size = len(tokenizer.word_index) + 1

In [ ]:
vocal_size

In [ ]:
max_length = max(len(caption.split()) for caption in all_captions)
max_length

Train Test Split

In [ ]:
image_ids = list(mapping.keys())
split = int(len(image_ids) * 0.85)
train = image_ids[:split]
test = image_ids[split:]
split


In [ ]:
len(test)

In [ ]:
#create data generator to get data in batch (avoid crash)
def data_generator(data_keys, mapping, features, tokenizer, max_length, vocal_size, batch_size):
    X1, X2, y = list(), list(), list()
    n = 0
    while 1:
        for key in data_keys:
            n+=1
            captions = mapping[key]
            #process each caption
            for caption in captions:
                #encode the sequence
                seq = tokenizer.texts_to_sequences([caption])[0]
                #split the seq to X, y pair
                for i in range(1, len(seq)):
                    #split into input and output pair
                    in_seq, out_seq= seq[:i], seq[i]
                    #pad input seq
                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                    #encode output seq
                    out_seq = to_categorical([out_seq], num_classes=vocal_size)[0]
                    #store the seq
                    X1.append(features[key][0])
                    X2.append(in_seq)
                    y.append(out_seq)
            if n == batch_size:
                X1, X2, y = np.array(X1, dtype=np.float32), np.array(X2, dtype=np.int32), np.array(y, dtype=np.float32)
                yield {"image": X1, "text": X2}, y
                X1, X2, y = list(), list(), list()
                n=0

Model Creation

In [ ]:
#image feature layer
inputs1 = Input(shape=(4096,),name="image")
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)
#sequence feature layer
inputs2 = Input(shape=(max_length,), name="text")
se1 = Embedding(vocal_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)

#decoder model
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocal_size, activation='softmax')(decoder2)

model = Model(inputs = [inputs1, inputs2], outputs= outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')

plot_model(model, show_shapes=True)

In [ ]:
#train model
epochs = 5
batch_size = 32
steps = len(train) // batch_size
for i in range(epochs):
    #create data generator
    generator = data_generator(train, mapping, features, tokenizer, max_length, vocal_size, batch_size)
    model.fit(generator, epochs = 1, steps_per_epoch=steps, verbose = 1)

In [ ]:
model.save(WORKING_DIR+'/best_model.h5')

Generate Captions for the Image

In [ ]:
def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

In [ ]:
# generate caption for an image
def predict_caption(model, image, tokenizer, max_length):
    #add start tag for generation process
    in_text = 'startseq'
    # iterate over the max length of sequence
    for i in range(max_length):
        # encode input sequence
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        # pad the sequences
        sequence = pad_sequences([sequence], max_length, padding = 'post')
        # predict next word
        yhat = model.predict([image, sequence], verbose=0)
        # get index with high probability
        yhat = np.argmax(yhat)
        # convert index to word
        word = idx_to_word(yhat, tokenizer)
        # stop if word not found
        if word is None:
            break
        #append word as input for generating next word
        in_text += " " + word
        # stop if reach end tag
        if word == 'endseq':
            break
    return in_text


In [ ]:
from nltk.translate.bleu_score import corpus_bleu
# validate with test data
actual, predicted = list(), list()
bleu1, bleu2, bleu3, bleu4 = 0 ,0,0,0
for key in tqdm(test):
    # get actual caption
    captions = mapping[key]
    # predict the caption for image
    y_pred = predict_caption(model, features[key], tokenizer, max_length)
    # split into words
    actual_captions = [caption.split() for caption in captions]
    y_pred = y_pred.split()
    # append to the list
    actual.append(actual_captions)
    predicted.append(y_pred)
    # calcuate BLEU score
    # print("BLEU-1: %f" % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
    # print("BLEU-2: %f" % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))
    # print("BLEU-3: %f" % corpus_bleu(actual, predicted, weights=(1/3, 1/3, 1/3, 0)))
    # print("BLEU-4: %f" % corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25)))
    bleu1+=corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0))
    bleu2+=corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0))
    bleu3+=corpus_bleu(actual, predicted, weights=(1/3,1/3,1/3, 0))
    bleu4+=corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))
print("BLEU-1: %f", bleu1/len(test))
print("BLEU-2: %f", bleu2/len(test))
print("BLEU-3: %f", bleu3/len(test))
print("BLEU-4: %f", bleu4/len(test))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
def generate_caption(image_name):
    # load the image
    # image_name = "1001773457_577c3a7d70.jpg"
    image_id = image_name.split('.')[0]
    img_path = os.path.join(BASE_DIR, "Images", image_name)
    image = Image.open(img_path)
    captions = mapping[image_id]
    print('---------------------Actual---------------------')
    for caption in captions:
        print(caption)
    # predict the caption
    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)
    print('--------------------Predicted--------------------')
    print(y_pred)
    plt.imshow(image)

Predict any image

In [ ]:
def load_feature(link_img, img_name):
        img_path =  link_img + '/' + img_name
        image = load_img(img_path, target_size=(224,224))
        #convert img to np.array
        image = img_to_array(image)
        #reshape
        image= image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
        #preprocess
        image = preprocess_input(image)
        # extract feature
        model = VGG16()
        model = Model(inputs=model.inputs, outputs=model.layers[-2].output)
        feature = model.predict(image, verbose=0)
        return feature
def predict_any_cap(img_name):
    link_img = '/content/sample_data/image'
    feature_cap = load_feature(link_img, img_name)
    y_pred = predict_caption(model, feature_cap, tokenizer, max_length)
    print('--------------------Predicted--------------------')
    print(y_pred)
    img_path = os.path.join(link_img,img_name)
    image = Image.open(img_path)
    plt.imshow(image)

In [ ]:
predict_any_cap("chupanh.jpg")

NameError: name 'predict_any_cap' is not defined

In [ ]:
generate_caption("/content/sample_data/img_cap/Images/1000268201_693b08cb0e.jpg")
